# 02 · Retrieval strategies

Lexical retrieval is a strong baseline but it only matches surface terms. This
notebook compares the three retrievers the package ships with and scores them
on a golden dataset using the same metrics the rest of the system reports.

- **BM25** — lexical term matching.
- **Vector** — cosine similarity over embeddings (deterministic fake embedder
  here so the notebook runs without model downloads).
- **Hybrid** — a weighted blend of the two.

In [1]:
import json
from pathlib import Path

import pandas as pd

from ragops_lab.ingestion import ChunkingConfig, ingest_directory, load_chunks_jsonl
from ragops_lab.retrieval import (
    BM25Retriever,
    FakeEmbeddingClient,
    HybridRetriever,
    RetrievalGoldenExample,
    VectorRetriever,
    evaluate_retrieval,
)

pd.set_option("display.max_colwidth", 80)
DATA = Path("../data")

## 1. Build the three retrievers over the same chunks

In [2]:
chunks_path = DATA / "processed" / "chunks.jsonl"
ingest_directory(
    DATA / "sample_documents",
    chunks_path,
    ChunkingConfig(chunk_size=220, overlap=20),
)
chunks = load_chunks_jsonl(chunks_path)

lexical = BM25Retriever(chunks)
vector = VectorRetriever(chunks, FakeEmbeddingClient())
hybrid = HybridRetriever(lexical, vector, lexical_weight=0.5, vector_weight=0.5)

retrievers = {"lexical": lexical, "vector": vector, "hybrid": hybrid}
list(retrievers)

['lexical', 'vector', 'hybrid']

## 2. Compare top hits for one query

Each retriever returns the same `RetrievalResult` shape, so the calling code
never changes — only the ranking strategy does.

In [3]:
query = "Which metrics are useful for RAG evaluation?"
rows = []
for name, retriever in retrievers.items():
    for r in retriever.search(query, top_k=2):
        rows.append(
            {
                "retriever": name,
                "rank": r.rank,
                "chunk_id": r.chunk.chunk_id,
                "score": round(r.score, 3),
                "preview": r.chunk.text[:60],
            }
        )
pd.DataFrame(rows)

,retriever,rank,chunk_id,score,preview
0,lexical,1,rag-evaluation:0,4.501,RAG evaluation should measure retrieval quality and answer q
1,lexical,2,readme:0,1.267,# Sample documents\n\nPlace small public-domain or synthetic d
2,hybrid,1,rag-evaluation:0,0.500,RAG evaluation should measure retrieval quality and answer q
3,hybrid,2,readme:0,0.141,# Sample documents\n\nPlace small public-domain or synthetic d


## 3. Score every strategy on the golden set

`data/golden/qa.json` pairs queries with the chunk ids that should be
retrieved. `evaluate_retrieval` computes recall@k and mean reciprocal rank;
because all three retrievers share the `search` interface, the same evaluation
function works for each.

In [4]:
golden = [RetrievalGoldenExample(**row) for row in json.loads((DATA / "golden" / "qa.json").read_text())]

report = []
for name, retriever in retrievers.items():
    metrics = evaluate_retrieval(retriever, golden, top_k=3)
    report.append(
        {
            "retriever": name,
            "recall@3": round(metrics.recall_at_k, 3),
            "MRR": round(metrics.mean_reciprocal_rank, 3),
        }
    )
pd.DataFrame(report).set_index("retriever")

,recall@3,MRR
retriever,,
lexical,1.0,1.0
vector,0.0,0.0
hybrid,1.0,1.0


On this tiny corpus all strategies recover the relevant chunks, but the harness
is what matters: swapping in real embeddings (`SentenceTransformerEmbeddingClient`)
or a larger corpus changes only the retriever construction — the evaluation,
CLI, and API stay identical.

Next: [`03_grounded_generation`](03_grounded_generation.ipynb) turns retrieved
context into a cited answer.